In [ ]:
import os
import random
from pathlib import Path
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset, TensorDataset
from tqdm import tqdm

from use_croma import PretrainedCROMA # from https://github.com/antofuller/CROMA


random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

eurosat_root = "./MSEuroSAT"

image_resolution = 120
feature_batch_size = 64
probe_batch_size = 1024

use_8_bit = False
num_workers = 4
pin_memory = torch.cuda.is_available()


class EuroSATMultispectralDataset(Dataset):
    """
    EuroSAT multispectral dataset loader for GeoTIFF files.
    """

    def __init__(
        self,
        root: str,
        image_size: int = 120,
        selected_classes: list[str] | None = None,
    ) -> None:
        """
        Initialize the dataset.

        Args:
            root: Dataset root directory.
            image_size: Output spatial resolution.
            selected_classes: Optional subset of class names.
        """
        self.root = Path(root)
        self.image_size = image_size

        all_classes = sorted(
            [
                path.name
                for path in self.root.iterdir()
                if path.is_dir()
            ]
        )

        if selected_classes is not None:
            self.class_names = [
                class_name
                for class_name in all_classes
                if class_name in selected_classes
            ]
        else:
            self.class_names = all_classes

        self.class_to_idx = {
            class_name: index
            for index, class_name in enumerate(self.class_names)
        }

        self.samples: list[tuple[Path, int]] = []

        for class_name in self.class_names:
            class_dir = self.root / class_name

            for tif_path in sorted(class_dir.glob("*.tif")):
                self.samples.append(
                    (
                        tif_path,
                        self.class_to_idx[class_name],
                    )
                )

        if not self.samples:
            raise RuntimeError(
                f"No .tif files found under {self.root}"
            )

        self.keep_indices = [
            0,
            1,
            2,
            3,
            4,
            5,
            6,
            7,
            8,
            9,
            11,
            12,
        ]

    def __len__(self) -> int:
        """
        Return the number of samples.
        """
        return len(self.samples)

    @staticmethod
    def normalize_batch(
        images: torch.Tensor,
        use_8_bit: bool,
    ) -> torch.Tensor:
        """
        Normalize a Sentinel image batch using CROMA normalization.

        Args:
            images: Tensor of shape [B, C, H, W].
            use_8_bit: Whether to return uint8 values.

        Returns:
            Normalized image tensor.
        """
        images = images.float()
        normalized_channels = []

        for channel in range(images.shape[1]):
            channel_values = images[:, channel, :, :]

            min_value = (
                channel_values.mean()
                - 2 * channel_values.std()
            )

            max_value = (
                channel_values.mean()
                + 2 * channel_values.std()
            )

            denominator = torch.clamp(
                max_value - min_value,
                min=1e-6,
            )

            normalized = (
                channel_values - min_value
            ) / denominator

            if use_8_bit:
                normalized = normalized * 255.0
                normalized = torch.clip(
                    normalized,
                    0,
                    255,
                )
                normalized = (
                    normalized
                    .unsqueeze(dim=1)
                    .to(torch.uint8)
                )
            else:
                normalized = torch.clip(
                    normalized,
                    0,
                    1,
                )
                normalized = normalized.unsqueeze(dim=1)

            normalized_channels.append(normalized)

        return torch.cat(
            normalized_channels,
            dim=1,
        )

    def __getitem__(
        self,
        index: int,
    ) -> dict[str, torch.Tensor | int | str]:
        """
        Load one sample.
        """
        tif_path, label = self.samples[index]

        with rasterio.open(tif_path) as src:
            image = src.read().astype(np.float32)

        image = torch.from_numpy(image)

        image = image[self.keep_indices]

        image = image.unsqueeze(0)

        image = F.interpolate(
            image,
            size=(
                self.image_size,
                self.image_size,
            ),
            mode="bilinear",
            align_corners=False,
        )

        image = image.squeeze(0)

        return {
            "image": image,
            "label": label,
            "path": str(tif_path),
        }


dataset = EuroSATMultispectralDataset(
    root=eurosat_root,
    image_size=image_resolution,
    selected_classes=[
        "Industrial",
        "Residential",
    ],
)

loader = DataLoader(
    dataset,
    batch_size=feature_batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)


@torch.no_grad()
def extract_features(
    backbone_model: nn.Module,
    data_loader: DataLoader,
    device_obj: torch.device,
    feature_key: str = "optical_GAP",
    use_8_bit_inputs: bool = True,
    l2_normalize_features: bool = False,
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Extract frozen CROMA embeddings.

    Args:
        backbone_model: Pretrained CROMA backbone.
        data_loader: DataLoader over EuroSAT.
        device_obj: Torch device.
        feature_key: CROMA feature key.
        use_8_bit_inputs: Whether to use uint8 inputs.
        l2_normalize_features: Whether to normalize embeddings.

    Returns:
        Feature array, label array, and sample paths.
    """
    backbone_model.eval()

    all_features = []
    all_labels = []
    all_paths = []

    for batch in tqdm(
        data_loader,
        desc="Extracting features",
    ):
        images = batch["image"].to(
            device_obj,
            non_blocking=True,
        )

        images = dataset.normalize_batch(
            images=images,
            use_8_bit=use_8_bit_inputs,
        )

        output = backbone_model(
            SAR_images=None,
            optical_images=images,
        )

        batch_features = output[feature_key]

        if l2_normalize_features:
            batch_features = F.normalize(
                batch_features,
                p=2,
                dim=1,
            )

        all_features.append(
            batch_features.float().cpu()
        )
        all_labels.append(
            batch["label"].cpu()
        )
        all_paths.extend(batch["path"])

    features_array = (
        torch.cat(all_features, dim=0)
        .numpy()
    )

    labels_array = (
        torch.cat(all_labels, dim=0)
        .numpy()
    )

    return (
        features_array,
        labels_array,
        all_paths,
    )


class MLPProbe(nn.Module):
    """
    Two-layer MLP probe.
    """

    def __init__(
        self,
        input_dim: int,
        num_classes: int,
        hidden_dim: int = 2048,
    ) -> None:
        """
        Initialize the probe.
        """
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        """
        Compute probe predictions.
        """
        return self.network(x)


class WarmupCosineScheduler:
    """
    Warmup followed by cosine learning-rate decay.
    """

    def __init__(
        self,
        optimizer: torch.optim.Optimizer,
        base_lr: float,
        warmup_epochs: int,
        total_epochs: int,
    ) -> None:
        """
        Initialize the scheduler.
        """
        self.optimizer = optimizer
        self.base_lr = base_lr
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs

    def step(
        self,
        epoch: int,
    ) -> None:
        """
        Update the learning rate.
        """
        if epoch < self.warmup_epochs:
            lr = self.base_lr * (
                epoch + 1
            ) / self.warmup_epochs
        else:
            progress = (
                epoch - self.warmup_epochs
            ) / max(
                self.total_epochs
                - self.warmup_epochs,
                1,
            )

            lr = (
                0.5
                * self.base_lr
                * (
                    1
                    + np.cos(
                        np.pi * progress
                    )
                )
            )

        for param_group in self.optimizer.param_groups:
            param_group["lr"] = lr


def train_one_probe(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    y_val: np.ndarray,
    lr: float,
    epochs: int = 105,
    warmup_epochs: int = 5,
    batch_size: int = 1024,
    weight_decay: float = 0.01,
    hidden_dim: int = 2048,
    use_bfloat16: bool = True,
    seed: int = 42,
) -> dict[str, float]:
    """
    Train one MLP probe.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    train_dataset = TensorDataset(
        torch.tensor(
            x_train,
            dtype=torch.float32,
        ),
        torch.tensor(
            y_train,
            dtype=torch.long,
        ),
    )

    val_dataset = TensorDataset(
        torch.tensor(
            x_val,
            dtype=torch.float32,
        ),
        torch.tensor(
            y_val,
            dtype=torch.long,
        ),
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    model = MLPProbe(
        input_dim=x_train.shape[1],
        num_classes=len(np.unique(y_train)),
        hidden_dim=hidden_dim,
    ).to(device)

    optimizer = AdamW(
        model.parameters(),
        lr=lr,
        betas=(0.9, 0.999),
        weight_decay=weight_decay,
    )

    scheduler = WarmupCosineScheduler(
        optimizer=optimizer,
        base_lr=lr,
        warmup_epochs=warmup_epochs,
        total_epochs=epochs,
    )

    autocast_enabled = (
        use_bfloat16
        and device.type == "cuda"
    )

    for epoch in range(epochs):
        model.train()
        scheduler.step(epoch)

        for (
            batch_features,
            batch_labels,
        ) in train_loader:
            batch_features = batch_features.to(
                device,
                non_blocking=True,
            )

            batch_labels = batch_labels.to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.bfloat16,
                enabled=autocast_enabled,
            ):
                logits = model(batch_features)

                loss = F.cross_entropy(
                    logits,
                    batch_labels,
                )

            loss.backward()
            optimizer.step()

    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for (
            batch_features,
            batch_labels,
        ) in val_loader:
            batch_features = batch_features.to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.bfloat16,
                enabled=autocast_enabled,
            ):
                logits = model(batch_features)

            predictions = (
                logits
                .argmax(dim=1)
                .cpu()
            )

            all_predictions.append(predictions)
            all_targets.append(batch_labels)

    predictions_array = (
        torch.cat(all_predictions)
        .numpy()
    )

    targets_array = (
        torch.cat(all_targets)
        .numpy()
    )

    accuracy = accuracy_score(
        targets_array,
        predictions_array,
    )

    macro_f1 = f1_score(
        targets_array,
        predictions_array,
        average="macro",
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
    }


def cross_validate_mlp_probe(
    features_array: np.ndarray,
    labels_array: np.ndarray,
    lr: float,
    n_splits: int = 5,
    epochs: int = 105,
    warmup_epochs: int = 5,
    batch_size: int = 1024,
    weight_decay: float = 0.01,
    hidden_dim: int = 2048,
    use_bfloat16: bool = True,
    seed: int = 42,
) -> list[dict[str, float]]:
    """
    Run stratified cross-validation with MLP probing.
    """
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    fold_results = []

    for fold, (
        train_idx,
        val_idx,
    ) in enumerate(
        skf.split(
            features_array,
            labels_array,
        ),
        start=1,
    ):
        print(f"\n===== Fold {fold} =====")

        x_train = features_array[train_idx]
        x_val = features_array[val_idx]

        y_train = labels_array[train_idx]
        y_val = labels_array[val_idx]

        result = train_one_probe(
            x_train=x_train,
            y_train=y_train,
            x_val=x_val,
            y_val=y_val,
            lr=lr,
            epochs=epochs,
            warmup_epochs=warmup_epochs,
            batch_size=batch_size,
            weight_decay=weight_decay,
            hidden_dim=hidden_dim,
            use_bfloat16=use_bfloat16,
            seed=seed + fold,
        )

        print(
            f"Fold {fold}: "
            f"lr={lr:.0e} "
            f"accuracy="
            f"{result['accuracy']:.4f} "
            f"f1_macro="
            f"{result['macro_f1']:.4f}"
        )

        fold_results.append(
            {
                "fold": fold,
                "lr": lr,
                "accuracy": result["accuracy"],
                "macro_f1": result["macro_f1"],
            }
        )

    accuracies = np.array(
        [
            result["accuracy"]
            for result in fold_results
        ]
    )

    macro_f1_scores = np.array(
        [
            result["macro_f1"]
            for result in fold_results
        ]
    )

    print(
        "\n=== Final cross-validation results ==="
    )

    print(
        f"Accuracy: "
        f"mean={accuracies.mean():.4f} "
        f"std={accuracies.std():.4f}"
    )

    print(
        f"F1-macro: "
        f"mean={macro_f1_scores.mean():.4f} "
        f"std={macro_f1_scores.std():.4f}"
    )

    return fold_results


def summarize_cv_results(
    fold_results: list[dict[str, float]],
) -> dict[str, float]:
    """
    Compute mean and standard deviation across folds.
    """
    accuracies = np.array(
        [
            result["accuracy"]
            for result in fold_results
        ]
    )

    f1_scores = np.array(
        [
            result["macro_f1"]
            for result in fold_results
        ]
    )

    return {
        "accuracy_mean": accuracies.mean(),
        "accuracy_std": accuracies.std(),
        "f1_macro_mean": f1_scores.mean(),
        "f1_macro_std": f1_scores.std(),
    }


def reinit_weights(
    module: torch.nn.Module,
) -> None:
    """
    Reinitialize supported CROMA layers.
    """
    if isinstance(
        module,
        torch.nn.Linear,
    ):
        torch.nn.init.xavier_uniform_(
            module.weight
        )

        if module.bias is not None:
            torch.nn.init.zeros_(
                module.bias
            )

    elif isinstance(
        module,
        torch.nn.LayerNorm,
    ):
        torch.nn.init.ones_(
            module.weight
        )

        torch.nn.init.zeros_(
            module.bias
        )


# ---------------------------------------------------------------------
# Model configuration.
# ---------------------------------------------------------------------

MODEL_PATHS = {
    "Full model": "./croma_epoch_299_converted.pt",
    "Selective training": (
        "./checkpoints/"
        "croma_30pct_one_step_converted.pt"
    ),
    "Scheduled training": (
        "./checkpoints/"
        "croma_30pct_stage_02_end_converted.pt"
    ),
    "Selective finetuning": (
        "./checkpoints/"
        "croma_30pct_finetuning_converted.pt"
    ),
    "Very selective finetuning": (
        "./checkpoints/"
        "croma_3pct_finetuning_converted.pt"
    ),
}
 
PROBE_LR = 5e-2
N_SPLITS = 5


def evaluate_backbone(
    model_name: str,
    pretrained_path: str,
) -> dict[str, float]:
    """
    Extract EuroSAT features and evaluate them with the MLP probe.

    Args:
        model_name: Human-readable model name.
        pretrained_path: Path to the converted CROMA checkpoint.

    Returns:
        Cross-validation summary containing accuracy and macro F1 metrics.
    """
    print(f"\n=== {model_name} ===")

    backbone = PretrainedCROMA(
        pretrained_path=pretrained_path,
        size="base",
        modality="optical",
        image_resolution=image_resolution,
    ).to(device)

    backbone.eval()

    features, labels, _ = extract_features(
        backbone_model=backbone,
        data_loader=loader,
        device_obj=device,
        feature_key="optical_GAP",
        use_8_bit_inputs=use_8_bit,
        l2_normalize_features=False,
    )

    fold_results = cross_validate_mlp_probe(
        features_array=features,
        labels_array=labels,
        lr=PROBE_LR,
        n_splits=N_SPLITS,
        batch_size=probe_batch_size,
    )

    summary = summarize_cv_results(fold_results)

    del backbone
    del features

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


def evaluate_random_backbone(
    pretrained_path: str,
    random_state: int = 42,
) -> dict[str, float]:
    """
    Evaluate a randomly reinitialized CROMA backbone.

    The checkpoint is loaded only to instantiate the expected architecture,
    after which supported trainable layers are reinitialized.

    Args:
        pretrained_path: Path used to instantiate the CROMA architecture.
        random_state: Seed used for reproducible weight initialization.

    Returns:
        Cross-validation summary containing accuracy and macro F1 metrics.
    """
    print("\n=== Random model ===")

    random.seed(random_state)
    np.random.seed(random_state)
    torch.manual_seed(random_state)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_state)

    backbone = PretrainedCROMA(
        pretrained_path=pretrained_path,
        size="base",
        modality="optical",
        image_resolution=image_resolution,
    ).to(device)

    backbone.apply(reinit_weights)
    backbone.eval()

    features, labels, _ = extract_features(
        backbone_model=backbone,
        data_loader=loader,
        device_obj=device,
        feature_key="optical_GAP",
        use_8_bit_inputs=use_8_bit,
        l2_normalize_features=False,
    )

    fold_results = cross_validate_mlp_probe(
        features_array=features,
        labels_array=labels,
        lr=PROBE_LR,
        n_splits=N_SPLITS,
        batch_size=probe_batch_size,
    )

    summary = summarize_cv_results(fold_results)

    del backbone
    del features

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


# ---------------------------------------------------------------------
# Evaluate all requested model variants.
# ---------------------------------------------------------------------

full_model_baseline = evaluate_backbone(
    model_name="Full model",
    pretrained_path=MODEL_PATHS["Full model"],
)

random_baseline = evaluate_random_backbone(
    pretrained_path=MODEL_PATHS["Full model"],
    random_state=42,
)

selective_training = evaluate_backbone(
    model_name="Selective training",
    pretrained_path=MODEL_PATHS["Selective training"],
)

scheduled_training = evaluate_backbone(
    model_name="Scheduled training",
    pretrained_path=MODEL_PATHS["Scheduled training"],
)

selective_finetuning = evaluate_backbone(
    model_name="Selective finetuning",
    pretrained_path=MODEL_PATHS["Selective finetuning"],
)

very_selective_finetuning = evaluate_backbone(
    model_name="Very selective finetuning",
    pretrained_path=MODEL_PATHS["Very selective finetuning"],
)


# ---------------------------------------------------------------------
# Print results 
# ---------------------------------------------------------------------

print("\n=== Baselines ===")

print(
    "Full model - "
    f"F1={full_model_baseline['f1_macro_mean']:.4f} "
    f"+/- {full_model_baseline['f1_macro_std']:.4f}"
)

print(
    "Random model - "
    f"F1={random_baseline['f1_macro_mean']:.4f} "
    f"+/- {random_baseline['f1_macro_std']:.4f}"
)

print("\n=== Selective models ===")

print(
    "Selective model - "
    f"F1={selective_training['f1_macro_mean']:.4f} "
    f"+/- {selective_training['f1_macro_std']:.4f}"
)

print(
    "Selective scheduled model - "
    f"F1={scheduled_training['f1_macro_mean']:.4f} "
    f"+/- {scheduled_training['f1_macro_std']:.4f}"
)

print(
    "Selective finetuned model - "
    f"F1={selective_finetuning['f1_macro_mean']:.4f} "
    f"+/- {selective_finetuning['f1_macro_std']:.4f}"
)

print(
    "Very selective finetuned model - "
    f"F1={very_selective_finetuning['f1_macro_mean']:.4f} "
    f"+/- {very_selective_finetuning['f1_macro_std']:.4f}"
)

In [ ]:
# ---------------------------------------------------------------------
# Comparative plot.
# ---------------------------------------------------------------------

import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/truetype/cmu/cmunss.ttf"
font_manager.fontManager.addfont(font_path)
font_prop = font_manager.FontProperties(fname=font_path)
plt.rcParams["font.family"] = font_prop.get_name()


results = {
    "Full model": full_model_baseline,
    "Random model": random_baseline,
    "Selective training (30%)": selective_training,
    "Scheduled selective training (30%)": scheduled_training,
    "Selective finetuning (30%)": selective_finetuning,
    "Very selective finetuning (3%)": very_selective_finetuning,
}

model_names = list(results.keys())

f1_means = [
    results[name]["f1_macro_mean"]
    for name in model_names
]

f1_stds = [
    results[name]["f1_macro_std"]
    for name in model_names
]

fig, ax = plt.subplots(figsize=(5, 8))

bars = ax.bar(
    model_names,
    f1_means,
    yerr=f1_stds,
    capsize=5,
)

ax.set_ylabel("Macro F1-score")
ax.set_title("EuroSAT classification (residential vs industrial)")
ax.set_ylim(0.4, 1.05)

ax.tick_params(
    axis="x",
    rotation=90,
)

for bar, mean in zip(bars, f1_means):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{mean:.3f}",
        ha="center",
        va="bottom",
    )

fig.tight_layout()
plt.savefig(
    "eurosat_model_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
import json

plot_results = {
    model_name: {
        "f1_mean": float(metrics["f1_macro_mean"]),
        "f1_std": float(metrics["f1_macro_std"]),
    }
    for model_name, metrics in results.items()
}

with open("eurosat_results.json", "w", encoding="utf-8") as file:
    json.dump(plot_results, file, indent=4)